In [1]:
import pandas as pd

# First peek — load only 5 rows to see structure
df_peek = pd.read_csv(r'C:\BFSI_Loan_Analytics\data\raw\lending_club_loans.csv', nrows=5)

print("Shape:", df_peek.shape)
print("\nColumn names:")
for i, col in enumerate(df_peek.columns):
    print(f"  {i+1}. {col}")

Shape: (5, 151)

Column names:
  1. id
  2. member_id
  3. loan_amnt
  4. funded_amnt
  5. funded_amnt_inv
  6. term
  7. int_rate
  8. installment
  9. grade
  10. sub_grade
  11. emp_title
  12. emp_length
  13. home_ownership
  14. annual_inc
  15. verification_status
  16. issue_d
  17. loan_status
  18. pymnt_plan
  19. url
  20. desc
  21. purpose
  22. title
  23. zip_code
  24. addr_state
  25. dti
  26. delinq_2yrs
  27. earliest_cr_line
  28. fico_range_low
  29. fico_range_high
  30. inq_last_6mths
  31. mths_since_last_delinq
  32. mths_since_last_record
  33. open_acc
  34. pub_rec
  35. revol_bal
  36. revol_util
  37. total_acc
  38. initial_list_status
  39. out_prncp
  40. out_prncp_inv
  41. total_pymnt
  42. total_pymnt_inv
  43. total_rec_prncp
  44. total_rec_int
  45. total_rec_late_fee
  46. recoveries
  47. collection_recovery_fee
  48. last_pymnt_d
  49. last_pymnt_amnt
  50. next_pymnt_d
  51. last_credit_pull_d
  52. last_fico_range_high
  53. last_fico_range

In [2]:
# Load full dataset — this will take 1-3 minutes
df = pd.read_csv(r'C:\BFSI_Loan_Analytics\data\raw\lending_club_loans.csv', low_memory=False)

print("Full dataset shape:", df.shape)
print("\nloan_status value counts:")
print(df['loan_status'].value_counts())

Full dataset shape: (2260701, 151)

loan_status value counts:
loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
Name: count, dtype: int64


In [3]:
# Filter to only Fully Paid and Charged Off
df_filtered = df[df['loan_status'].isin(['Fully Paid', 'Charged Off'])].copy()

print("Before filtering:", df.shape[0], "rows")
print("After filtering:", df_filtered.shape[0], "rows")
print("Rows removed:", df.shape[0] - df_filtered.shape[0])
print("\nloan_status value counts after filter:")
print(df_filtered['loan_status'].value_counts())

Before filtering: 2260701 rows
After filtering: 1345310 rows
Rows removed: 915391

loan_status value counts after filter:
loan_status
Fully Paid     1076751
Charged Off     268559
Name: count, dtype: int64


In [4]:
# Calculate null percentage for every column
null_summary = pd.DataFrame({
    'null_count': df_filtered.isnull().sum(),
    'null_percent': (df_filtered.isnull().sum() / len(df_filtered) * 100).round(2)
})

# Show only columns that actually have nulls, sorted worst first
null_summary = null_summary[null_summary['null_count'] > 0].sort_values('null_percent', ascending=False)

print(f"Total columns with nulls: {len(null_summary)} out of {df_filtered.shape[1]}")
print("\n", null_summary.to_string())

Total columns with nulls: 105 out of 151

                                             null_count  null_percent
member_id                                      1345310        100.00
next_pymnt_d                                   1345310        100.00
orig_projected_additional_accrued_interest     1341551         99.72
hardship_type                                  1339556         99.57
hardship_reason                                1339556         99.57
hardship_status                                1339556         99.57
deferral_term                                  1339556         99.57
hardship_amount                                1339556         99.57
hardship_start_date                            1339556         99.57
hardship_end_date                              1339556         99.57
payment_plan_start_date                        1339556         99.57
hardship_length                                1339556         99.57
hardship_dpd                                   1339556      

In [5]:
# Drop all columns with more than 50% null values
threshold = 50.0
cols_to_drop = null_summary[null_summary['null_percent'] > threshold].index.tolist()

print(f"Columns to drop (>50% null): {len(cols_to_drop)}")
print(cols_to_drop)

df_filtered.drop(columns=cols_to_drop, inplace=True)

print(f"\nShape after dropping high-null columns: {df_filtered.shape}")

Columns to drop (>50% null): 58
['member_id', 'next_pymnt_d', 'orig_projected_additional_accrued_interest', 'hardship_type', 'hardship_reason', 'hardship_status', 'deferral_term', 'hardship_amount', 'hardship_start_date', 'hardship_end_date', 'payment_plan_start_date', 'hardship_length', 'hardship_dpd', 'hardship_loan_status', 'hardship_payoff_balance_amount', 'hardship_last_payment_amount', 'sec_app_mths_since_last_major_derog', 'sec_app_revol_util', 'sec_app_chargeoff_within_12_mths', 'sec_app_open_act_il', 'sec_app_num_rev_accts', 'sec_app_open_acc', 'sec_app_mort_acc', 'sec_app_inq_last_6mths', 'sec_app_earliest_cr_line', 'sec_app_fico_range_high', 'sec_app_fico_range_low', 'sec_app_collections_12_mths_ex_med', 'revol_bal_joint', 'verification_status_joint', 'dti_joint', 'annual_inc_joint', 'debt_settlement_flag_date', 'settlement_status', 'settlement_date', 'settlement_amount', 'settlement_percentage', 'settlement_term', 'desc', 'mths_since_last_record', 'mths_since_recent_bc_dlq'

In [6]:
# Drop columns that are irrelevant to loan risk analysis
cols_irrelevant = [
    'id',           # just a row identifier
    'url',          # link to loan listing — no analytical value
    'title',        # free text loan title — redundant with 'purpose'
    'zip_code',     # too granular, we have addr_state
    'policy_code',  # only 1 unique value in this dataset
    'pymnt_plan',   # almost all 'n', no variance
    'hardship_flag' # almost all 'N', no variance
]

df_filtered.drop(columns=cols_irrelevant, inplace=True)

print(f"Shape after dropping irrelevant columns: {df_filtered.shape}")
print(f"\nRemaining columns: {df_filtered.shape[1]}")

Shape after dropping irrelevant columns: (1345310, 86)

Remaining columns: 86


In [7]:
# Re-check nulls on remaining columns
null_remaining = pd.DataFrame({
    'null_count': df_filtered.isnull().sum(),
    'null_percent': (df_filtered.isnull().sum() / len(df_filtered) * 100).round(2)
})

null_remaining = null_remaining[null_remaining['null_count'] > 0].sort_values('null_percent', ascending=False)

print(f"Columns still with nulls: {len(null_remaining)}")
print("\n", null_remaining.to_string())

Columns still with nulls: 45

                             null_count  null_percent
mths_since_recent_inq           174071         12.94
num_tl_120dpd_2m                117401          8.73
mo_sin_old_il_acct              105575          7.85
emp_title                        85785          6.38
emp_length                       78511          5.84
pct_tl_nvr_dlq                   67681          5.03
num_actv_bc_tl                   67527          5.02
mo_sin_rcnt_tl                   67527          5.02
tot_hi_cred_lim                  67527          5.02
num_tl_op_past_12m               67527          5.02
num_tl_90g_dpd_24m               67527          5.02
num_tl_30dpd                     67527          5.02
num_rev_tl_bal_gt_0              67527          5.02
num_rev_accts                    67528          5.02
num_op_rev_tl                    67527          5.02
num_il_tl                        67527          5.02
num_bc_tl                        67527          5.02
num_actv_rev_tl

In [8]:
# 1. Fill numeric columns with median
numeric_fill_median = [
    'mths_since_recent_inq', 'num_tl_120dpd_2m', 'mo_sin_old_il_acct',
    'pct_tl_nvr_dlq', 'num_actv_bc_tl', 'mo_sin_rcnt_tl', 'tot_hi_cred_lim',
    'num_tl_op_past_12m', 'num_tl_90g_dpd_24m', 'num_tl_30dpd',
    'num_rev_tl_bal_gt_0', 'num_rev_accts', 'num_op_rev_tl', 'num_il_tl',
    'num_bc_tl', 'num_actv_rev_tl', 'num_accts_ever_120_pd',
    'total_il_high_credit_limit', 'total_rev_hi_lim', 'mo_sin_rcnt_rev_tl_op',
    'mo_sin_old_rev_tl_op', 'avg_cur_bal', 'tot_coll_amt', 'tot_cur_bal',
    'bc_util', 'percent_bc_gt_75', 'bc_open_to_buy', 'mths_since_recent_bc',
    'num_sats', 'num_bc_sats', 'total_bc_limit', 'total_bal_ex_mort',
    'mort_acc', 'acc_open_past_24mths', 'dti', 'revol_util'
]

for col in numeric_fill_median:
    df_filtered[col] = df_filtered[col].fillna(df_filtered[col].median())

# 2. Fill categorical columns with 'Unknown'
df_filtered['emp_length'] = df_filtered['emp_length'].fillna('Unknown')
df_filtered['emp_title'] = df_filtered['emp_title'].fillna('Unknown')

# 3. Fill count columns with 0
count_fill_zero = [
    'pub_rec_bankruptcies', 'tax_liens',
    'collections_12_mths_ex_med', 'chargeoff_within_12_mths', 'inq_last_6mths'
]
for col in count_fill_zero:
    df_filtered[col] = df_filtered[col].fillna(0)

# 4. Drop rows where last_pymnt_d or last_credit_pull_d is null
rows_before = len(df_filtered)
df_filtered = df_filtered.dropna(subset=['last_pymnt_d', 'last_credit_pull_d'])
rows_after = len(df_filtered)

print(f"Rows dropped due to null dates: {rows_before - rows_after}")
print(f"Shape after null handling: {df_filtered.shape}")
print(f"Any nulls remaining: {df_filtered.isnull().sum().sum()}")

Rows dropped due to null dates: 2368
Shape after null handling: (1342942, 86)
Any nulls remaining: 0


In [9]:
# Check data types of all columns
print(df_filtered.dtypes.to_string())

loan_amnt                     float64
funded_amnt                   float64
funded_amnt_inv               float64
term                           object
int_rate                      float64
installment                   float64
grade                          object
sub_grade                      object
emp_title                      object
emp_length                     object
home_ownership                 object
annual_inc                    float64
verification_status            object
issue_d                        object
loan_status                    object
purpose                        object
addr_state                     object
dti                           float64
delinq_2yrs                   float64
earliest_cr_line               object
fico_range_low                float64
fico_range_high               float64
inq_last_6mths                float64
open_acc                      float64
pub_rec                       float64
revol_bal                     float64
revol_util  

In [10]:
# Fix term — extract numeric value only
df_filtered['term'] = df_filtered['term'].str.replace('months', '').str.strip().astype(int)

# Fix date columns — convert to datetime
date_cols = ['issue_d', 'earliest_cr_line', 'last_pymnt_d', 'last_credit_pull_d']
for col in date_cols:
    df_filtered[col] = pd.to_datetime(df_filtered[col], format='%b-%Y')

# Verify fixes
print("term unique values:", df_filtered['term'].unique())
print("term dtype:", df_filtered['term'].dtype)
print("\nissue_d dtype:", df_filtered['issue_d'].dtype)
print("issue_d sample values:", df_filtered['issue_d'].head(3).tolist())

term unique values: [36 60]
term dtype: int32

issue_d dtype: datetime64[ns]
issue_d sample values: [Timestamp('2015-12-01 00:00:00'), Timestamp('2015-12-01 00:00:00'), Timestamp('2015-12-01 00:00:00')]


In [11]:
# Save cleaned dataset to processed folder
output_path = r'C:\BFSI_Loan_Analytics\data\processed\lending_club_cleaned.csv'
df_filtered.to_csv(output_path, index=False)

print(f"File saved to: {output_path}")
print(f"Final shape: {df_filtered.shape}")
print(f"Final row count: {df_filtered.shape[0]:,}")
print(f"Final column count: {df_filtered.shape[1]}")

File saved to: C:\BFSI_Loan_Analytics\data\processed\lending_club_cleaned.csv
Final shape: (1342942, 86)
Final row count: 1,342,942
Final column count: 86


In [12]:
import pandas as pd

df = pd.read_csv(r'C:\BFSI_Loan_Analytics\data\processed\lending_club_features.csv')

print("Total rows:", len(df))
print("Total defaults:", df['loan_outcome'].sum())
print("Default rate:", round(df['loan_outcome'].sum() / len(df) * 100, 2))

Total rows: 1342942
Total defaults: 266236
Default rate: 19.82
